# University Messy Dataset — Data Cleaning Practice Set

`master_dataset_raw.csv` — **1,005 rows × 15 columns**

This dataset is **deliberately messy**. Every quirk below was injected on
purpose so it maps to one exact step of the cleaning pipeline you outlined.
Nothing here is random noise you should ignore — each issue is something
to actively detect and fix.

---

## Column Reference

| Column                     | Type it *should* become | Messiness injected |
|---------------------------|--------------------------|---------------------|
| ` Student_ID `             | `string`                | Leading/trailing whitespace + mixed case in the **header itself**; ~3% of values are missing (unrecoverable → drop row) |
| `Full Name`                | `string`                | Clean (control column) |
| `Application Date`         | `datetime64`             | 5 different date string formats mixed together; ~3.5% missing |
| `Date of Birth`            | `datetime64`             | 3 different date string formats mixed together |
| `Age`                      | `Int64`                  | Impossible values injected (`-5`, `0`, `150`, `999`); ~4% missing |
| `Department`               | `category`/`string`      | Same real department spelled 4–5 different ways (typos, casing, double spaces) — e.g. `Entreprenuarship` / `Eentrepreneurship` / `entrepreneurship` |
| `Country`                  | `category`/`string`      | Clean (control column) |
| `Email`                    | `string`                 | Clean (control column) |
| `Fee_Paid`                 | `float`                  | Currency symbols mixed: `$4,843.15`, `Rs.3,810`, `2488.48 USD`, plain `2,136.02`; ~19% missing |
| `GPA`                      | `float`                  | A few values injected **above the 4.0 scale max** (5.0–9.9) — outliers; ~6% missing |
| `Attendance_Percentage`    | `float`                  | A few impossible percentages (`150.0`, `-20.0`, `105.5`); ~5% missing |
| `Is_Active`                | `bool`                   | Mixed representations: `True/False`, `TRUE/FALSE`, `1/0`, `Yes/No`, `yes/no`; ~11% missing |
| `Lab_Sensor_RPM`           | `float`                  | Simulated hardware-glitch outliers (`55000`, `48210`, `-100`, `99999`); ~3% missing |
| `Enrollment_Status`        | `category`/`string`      | ~30% missing/blank — a genuine categorical gap, ideal candidate for the `"Unknown"` flag technique |
| `Unnamed: 14`              | *(drop entirely)*        | 100% empty — a classic Excel "separator" column that should never survive Step 2 |

---

## What's baked in, mapped to your pipeline

**Step 1 — Profiling**
`df.shape` → `(1005, 15)`. `df.info()` will show every column as `object`
dtype even though several are really dates/numbers/booleans. `df.describe()`
on `Age`, `GPA`, `Attendance_Percentage`, and `Lab_Sensor_RPM` (after
forcing them numeric) will immediately flag the injected outliers.

**Step 2 — Structural standardization**
The ID column's header has stray spaces and mixed case on purpose
(` Student_ID `). `Unnamed: 14` is 100% null — a pure separator column
to drop.

**Step 3 — Deduplication**
30 rows are **exact** duplicates of existing rows. A further 25 rows
share a `Student_ID` with another row but have a different
`Application Date` / `Fee_Paid` — a realistic "resubmitted application"
partial-duplicate scenario, ideal for practicing `subset=` + `sort_values`
+ `keep='last'`.

**Step 4 — Type casting**
- `Application Date` / `Date of Birth`: 3–5 different string date formats mixed in the same column.
- `Fee_Paid`: currency symbols (`$`, `Rs.`), thousands separators, and trailing `"USD"` text.
- `Is_Active`: booleans represented as `True/False`, `1/0`, and `Yes/No` — all as strings.

**Step 5 — Missing values**
- `Student_ID` missing → these rows have no reliable key and should be **dropped**.
- `GPA` / `Attendance_Percentage` missing → good candidates for **mean/median imputation**.
- `Application Date` gaps → good candidate for **forward/backward fill** once sorted chronologically per student.
- `Enrollment_Status` missing (~30%) → flag as `"Unknown"` rather than dropping, since absence is itself meaningful here.

**Step 6 — Outliers & typos**
- `Age`: `-5`, `0`, `150`, `999` — impossible human ages.
- `GPA`: values above 4.0 on what should be a 4.0 scale.
- `Attendance_Percentage`: values above 100% or negative.
- `Lab_Sensor_RPM`: implausible sensor spikes simulating a hardware glitch.
- `Department`: the same real department deliberately spelled multiple ways — needs a mapping dictionary to consolidate into one canonical string per department.

**Step 7 — Final sanity check**
After you drop exact + partial duplicates and rows with missing
`Student_ID`, compare your final row count against the original 1,005 —
it should drop by roughly 30 (exact dupes) + up to 25 (partial dupes,
minus the ones you legitimately keep) + ~27 (missing IDs), landing
somewhere in the low 900s. Confirm zero unexpected nulls remain in any
column you chose to impute or flag, then export to `.csv` or `.parquet`.

---

## Suggested first commands to run

```python
import pandas as pd

df = pd.read_csv("master_dataset_raw.csv")

df.shape
df.info()
df.describe()
df.columns
```

From there, work through Steps 2–7 exactly as you outlined. Good luck —
this file should give you real reps on every technique in the pipeline.


## **Let's start Data Cleaning**(step-by-step)

**Step 1: Data Ingestion**

In [40]:
import pandas as pd
import numpy as np
import os

# File path.
file_path: str = '../messy_data_sets/university_messy_dataset.csv'

# Always check whether the file exists or not first.
if not os.path.exists(file_path):
    raise FileNotFoundError(f'Error! There is no such file {file_path}')
else:
    try:
        df: pd.DataFrame = pd.read_csv(
            file_path,
            # usecols = [list of columns] The memory Savior ingest only those columns which are necessary for the modeling ignore the rest.
            
            # dtype= {'column': datatype} This specifically use to say the pandas reader the data types of columns
            # NULL Standardization.
            na_values= [' ', '', 'Missing', 'missing', 'Unknown', 'unknown', 'N/A', 'NaN'],
            # Fault Tolerance
            on_bad_lines='warn', # If a row is corrupted (e.g., 6 columns instead of 5), skip it and log a warning instead of crashing.
            # PERFORMANCE
            engine= 'c' # Forces the underlying C-engine for maximum read speed.
        )
        print(f'Data Ingestion successful. Loaded {len(df)} rows into memory.')
    except Exception as e:
        print(f'PIPELINE FAILURE during CSV ingestion: {e}')


Data Ingestion successful. Loaded 1005 rows into memory.


**Step 2: Data Profiling and inspections**
- Check the structre of the data via using `df.shape()` method.
- Know about the dataset's pre assigned datatypes and non-null columns via `df.info()` mehtod.
- Get a quick statistical summary of numerical values in dataset via `df.info()` method.
- check the number of Null values in each columns via `df.isna().shape()`, however it may not be accurate, just because Null Values be actually Null. we humans read 'missing' 'unkown' 'NaN' as null values but in pandas it treats them as a valid and existing strings.

In [41]:
# number of rows and columns(sturcture of data).
dimensions: tuple = df.shape
print(f'The Dataset contains a total of {dimensions[0]} rows and {dimensions[1]} columns.')

The Dataset contains a total of 1005 rows and 15 columns.


In [42]:
# Let's check the dataset preassigned data types and non null values.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0    Student_ID            978 non-null    str    
 1   Full Name              1005 non-null   str    
 2   Application Date       970 non-null    str    
 3   Date of Birth          1005 non-null   str    
 4   Age                    963 non-null    float64
 5   Department             1005 non-null   str    
 6   Country                1005 non-null   str    
 7   Email                  1005 non-null   str    
 8   Fee_Paid               817 non-null    str    
 9   GPA                    943 non-null    float64
 10  Attendance_Percentage  954 non-null    float64
 11  Is_Active              898 non-null    str    
 12  Lab_Sensor_RPM         978 non-null    float64
 13  Enrollment_Status      551 non-null    str    
 14  Unnamed: 14            0 non-null      float64
dtypes: float64(5), 

In [43]:
# Let's get a quick summary of numerical values inside the dataset.
df.describe()

,Age,GPA,Attendance_Percentage,Lab_Sensor_RPM,Unnamed: 14
count,963.000000,943.000000,954.000000,978.000000,0.0
mean,31.593977,2.986299,70.620231,2639.137832,NaN
std,11.642110,1.261650,20.530368,6688.492448,NaN
min,-5.000000,1.500000,-20.000000,-100.000000,NaN
25%,24.000000,2.130000,55.625000,1462.525000,NaN
50%,32.000000,2.890000,69.950000,2109.100000,NaN
75%,38.000000,3.510000,85.800000,2628.325000,NaN
max,150.000000,9.700000,150.000000,99999.000000,NaN


In [44]:
# let's get the total number of actual null-values(according to pandas) in every columns.
df.isna().sum()

# Insights

""" We'll work this in further steps:

1. The student Id columns has null values, which it shouldn't , So I gotta drop every rows which
   doesn't contains the student Id, Since there' no need of such rows.
2. We does has an empty column named 'Unnamed: 14' gotta drop whole column.
"""

" We'll work this in further steps:\n\n1. The student Id columns has null values, which it shouldn't , So I gotta drop every rows which\n   doesn't contains the student Id, Since there' no need of such rows.\n2. We does has an empty column named 'Unnamed: 14' gotta drop whole column.\n"

**Step 3: Structural Standardization**
- strip all the empty whitespaces around the column ID's.
- Make every columns looks standardized either uppercase or lower cases with Underscore e.g., `STUDENT_ID`, `student_id`.
- Drop all the empty columns, which are present for human readability , whcich would be named 'Unnamed: 14'

In [54]:
# To strip all the empty white spaces we're gonna use the .str method.
# and so for standardizing them to uppercases and replace the seperator empty
# space to underscore.

# sometimes we gotta force our dataframe columns datatype to be string manually
# it may happen that any of the column ID would be in different type and using
# the .str method on other datatypes rather then strings, the program would crash.

# df.columns = df.columns.astype(str) # wouldn't be necessary on this particular dataset.

df.columns = df.columns.str.strip().str.upper().str.replace(' ', '_')

# let's drop the empty columns presented just for human readability.
df: pd.DataFrame = df.dropna(axis=1, how='all')
df.columns.tolist() # Final columns of dataframe perfectly standardized, which won't lead to any crashes.


['STUDENT_ID',
 'FULL_NAME',
 'APPLICATION_DATE',
 'DATE_OF_BIRTH',
 'AGE',
 'DEPARTMENT',
 'COUNTRY',
 'EMAIL',
 'FEE_PAID',
 'GPA',
 'ATTENDANCE_PERCENTAGE',
 'IS_ACTIVE',
 'LAB_SENSOR_RPM',
 'ENROLLMENT_STATUS']

**Step 4: Deduplication**
